In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

<font size="5" color="black">ch01. 머신러닝 입문</font>

In [ ]:
# GPU 사용 여부 확인
import tensorflow as tf
print(tf.__version__)
tf.config.list_physical_devices('GPU')

In [ ]:
# GPU 비활성화
import os
os.environ['CUDA_VISIBLE_DEVICES'] = "-1"

# 1. 기존의 프로그램 방식

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def celsius_to_faherenheit(c):
    return 1.8*c + 32

In [ ]:
input_data = int(input('섭씨 온도 : '))
print('화씨 온도 :', celsius_to_faherenheit(input_data))

# 2. 머신러닝/딥러닝 프로그램 방식

1. 데이터 확보 및 생성
2. 데이터 전처리 : 스케일 조정, 라벨링 처리, 데이터셋 분할 [훈련 데이터셋(학습데이터셋), 검증 데이터셋, 시험 데이터셋으로 나누기]
3. 모델 구성
4. 모델 학습과정 설정
5. 모델 학습시키기 (학습 데이터셋과 검증 데이터셋 사용)
6. 모델 평가 (시험 데이터셋 사용)
7. 모델 사용 (입력값을 통해 예측값 받기)

## 2.1 노이즈가 없는 데이터

### 1. 데이터 확보 및 생성

In [ ]:
data_C = np.arange(100)
data_C # 입력변수 == 독립변수

In [ ]:
data_F = celsius_to_faherenheit(data_C)
data_F # 타겟변수 == 종속변수

### 2. 데이터 전처리 : 스케일 조정 (Normalize)

In [ ]:
scaled_data_C = data_C / 100
scaled_data_F = data_F / 100

In [ ]:
print('독립변수\n', scaled_data_C)
print('타겟변수\n', scaled_data_F)

### 3. 모델 구성
- tensorflow

In [ ]:
from tensorflow.keras.models import Sequential # 모델 생성
from tensorflow.keras.layers import Dense, Input # 입력값과 출력값으로 layer층 지정
model = Sequential()
# Dense(전결합) 레이어 추가
model.add(Dense(1, # # 출력 node 개수: 타겟 변수가 1개
               input_shape=(1,) # 입력 데이터의 shape : 입력 특성(Feature)이 1개인 1차원 데이터
               ))
print(model.summary()) # Param # : 파라미터(가중치 W, 편향 b)의 총 개수

### 4. 모델 학습과정 설정
- 회귀분석 오차함수 : MSE (오차 제곱 평균), RMSE (오차 제곱 평균의 제곱근), MAE (절댓값 평균)

In [ ]:
model.compile(loss='mse', # 손실 함수 : 실제로 최소화해야 할 최우선 오차 목표
              optimizer='rmsprop', # 옵티마이저 : 가중치와 편향을 업데이트하는 알고리즘
              metrics=['mae'] # 평가지표 : 모니터링용 지표
             )

In [ ]:
# 학습 전 예측치
model.predict(np.array([[0],[0.01]])) 

In [ ]:
# 학습 전 모델 저장
model.save('model/before_learning.h5') # kernel 내에 w(기울기), bias 내에 b(y절편)이 저장됨

### 5. 모델 학습시키기

In [ ]:
hist = model.fit(scaled_data_C, # 독립변수
                 scaled_data_F, # 타겟변수
                 epochs=1000, # 학습 횟수
                 verbose=2) # 학습 과정의 출력을 제어

In [ ]:
hist.history.keys()

In [ ]:
plt.plot(hist.history['loss'], 'r', label='loss')
plt.plot(hist.history['mae'], 'b', label='mae')
plt.legend()
plt.xlabel('Epoch')
plt.ylabel('Gap')

### 6. 모델 사용하기

In [ ]:
# 학습 후 예측치
model.predict(np.array([[0],[0.01]]))

In [ ]:
# 학습 후 모델 저장
model.save('model/after_learning.h5') # w_hat = 1.7996441, b_hat = 0.31953412

In [ ]:
# 모델 불러오기
from tensorflow.keras.models import load_model
model = load_model('model/after_learning.h5')

In [ ]:
model.predict(np.array([[0],[0.01],[0.02]]))

In [ ]:
# 예측값 y_hat과 실제값 scaled_data_F 비교
y_hat = model.predict([scaled_data_C])

for h, y in zip(y_hat[:5], scaled_data_F[:5]):
    print(h, y)

In [ ]:
square_error = 0

for h, y in zip(y_hat, scaled_data_F):
    square_error += (h[0]-y)**2 # y_hat은 2차원 리스트 형태로, h[0]는 y_hat 리스트의 각 요소를 의미
print(square_error/100)

In [ ]:
plt.scatter(x=scaled_data_C, y=scaled_data_F, c='b', s=10)
plt.scatter(x=scaled_data_C, y=y_hat, c='r', s=10, alpha=0.5)
plt.xlabel('C')
plt.ylabel('F', rotation=0, labelpad=20)
plt.show()

## 2.2 노이즈가 있는 데이터

### 1~2 .데이터 생성 및 전처리

In [ ]:
np.random.seed(707)
noise = np.random.normal(scale = 0.1, size=100)
noised_scaled_data_F = noise + scaled_data_F

In [ ]:
plt.scatter(x=scaled_data_C, y=noised_scaled_data_F, c='b', s=10)
plt.xlabel('C')
plt.ylabel('F', rotation=0, labelpad=20)
plt.show()

### 3. 모델 구성

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
model2 = Sequential()
model2.add(Dense(units=1, input_shape=(1,)))

### 4. 모델 학습과정 설정

In [ ]:
model2.compile(loss='mse', # 손실 함수
               optimizer='rmsprop', # 옵티마이저 : 추론 알고리즘
               metrics=['mae']) # 평가 지표

### 5. 모델 학습시키기

In [ ]:
hist2 = model2.fit(scaled_data_C, noised_scaled_data_F, epochs = 1000, verbose=1)

In [ ]:
hist2.history.keys()

In [ ]:
plt.plot(hist2.history['loss'], 'r', label='LOSS')

### 6. 모델 사용하기

In [ ]:
y_hat = model2.predict([scaled_data_C])
for h, y in zip(y_hat[:5], noised_scaled_data_F[:5]):
    print(h, y)

In [ ]:
y_hat = model2.predict([scaled_data_C])
for h, y in zip(y_hat, noised_scaled_data_F):
    square_error += (h[0]-y)**2
print(square_error/100)

In [ ]:
plt.scatter(x=scaled_data_C, y=noised_scaled_data_F, c='b')
plt.scatter(x=scaled_data_C, y=y_hat, c='r')